<a href="https://colab.research.google.com/github/ritvik-123/114-Assignments-OS/blob/main/Final_LogReg.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [24]:
# --- Imports -----------------------------------------------------------
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer

from sklearn.model_selection import GroupKFold
from sklearn.decomposition import TruncatedSVD         # dimensionality reduction (SVD works with dense/sparse)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score
)

from sklearn.metrics import top_k_accuracy_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.model_selection import train_test_split

import torch

In [7]:
# --- Config --------------------------------------------------------------
CSV_PATH = "../Data/"

MODEL_PATH = "../Model/"   # your labeled sentence-level file

TARGET_LABELS = [                                # the 4 label columns we're predicting
    "ideological",
    "institutionalized",
    "interpersonal",
    "internalized",
]
EMBEDDING_MODEL_NAME = "BAAI/bge-large-en-v1.5"  # 384-dim model: smaller than bge-large, less overfitting risk
N_SVD_COMPONENTS = 50                            # compress 384-dim embeddings down further before the classifier
N_FOLDS = 5                                      # number of GroupKFold splits (you could also use LeaveOneGroupOut)
RANDOM_STATE = 42                                # fixed seed so results are reproducible
THRESHOLD = 0.5                                  # probability cutoff for turning a score into a 0/1 prediction

%pwd
%cd /content/drive/MyDrive/EMP/Notebook

/content/drive/MyDrive/EMP/Notebook


In [8]:
# --- Load -----------------------------------------------------------------
df = pd.read_csv(CSV_PATH + "Generated_Sentences_1.csv")

df.drop(columns = ['data_source'])

df_1 = pd.read_csv(CSV_PATH + "Module 1 Sentences Gemini.csv")

df_1 = df_1[['sentence','ideological','institutionalized','interpersonal','internalized','primary_leaning']]
df_1 = df_1[df_1['primary_leaning'] != 'none']
df_1 = df_1.reset_index(drop=True)
df_1 = df_1.rename(columns={'sentence':'Sentence'})
df_1 = df_1.rename(columns={'primary_leaning':'Label'})
df_1["data_source"] = "real world"

df = pd.concat([df, df_1], ignore_index=True)

# df = df[df['data_source'] != 'extra_institutionalized']

# Clean column names just in case there are spaces
df.columns = df.columns.str.strip()

print("Columns:", df.columns.tolist())

# Clean sentences
df["Sentence"] = df["Sentence"].fillna("").astype(str).str.strip()
df = df[df["Sentence"] != ""].reset_index(drop=True)

# Clean Label column
df["Label"] = df["Label"].fillna("").astype(str).str.strip().str.lower()

# Create the 4 target label columns from the single Label column
for label in TARGET_LABELS:
    df[label] = (df["Label"] == label).astype(int)

# Optional: create source_id if your new file does not have one
if "source_id" not in df.columns:
    df["source_id"] = df.index

print("Rows after cleaning:", len(df))
print("Unique groups (source_id):", df["source_id"].nunique())
print(df[TARGET_LABELS].sum())

Columns: ['Sentence', 'Label', 'data_source', 'ideological', 'institutionalized', 'interpersonal', 'internalized']
Rows after cleaning: 810
Unique groups (source_id): 810
ideological          203
institutionalized    256
interpersonal        195
internalized         156
dtype: int64


In [11]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


In [12]:
# --- Embed ------------------------------------------------------------------
device = "cuda"   # change to "cpu" if you're not on a GPU runtime

embedder = SentenceTransformer(EMBEDDING_MODEL_NAME, device=device)   # loads the pretrained embedding model

sentences = df["Sentence"].tolist()                                   # plain list of strings to embed

X_full = embedder.encode(
    sentences,                     # the sentences to embed
    batch_size=32,                 # how many sentences to embed per forward pass
    normalize_embeddings=True,     # L2-normalize each embedding (helps cosine-similarity-style comparisons)
    show_progress_bar=True,        # display a progress bar since this can take a minute
    convert_to_numpy=True,         # return a numpy array instead of torch tensors
)

print("Embedding matrix shape:", X_full.shape)   # expect (num_sentences, 384)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/779 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Batches:   0%|          | 0/26 [00:00<?, ?it/s]

Embedding matrix shape: (810, 1024)


In [15]:
# ------------------------------------------------------------
# Train/test split BEFORE SVD + Logistic Regression
# ------------------------------------------------------------

y_full = df["Label"].values

X_train_embed, X_test_embed, y_train, y_test, df_train, df_test = train_test_split(
    X_full,
    y_full,
    df,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_full
)

print("Train shape:", X_train_embed.shape)
print("Test shape:", X_test_embed.shape)

Train shape: (648, 1024)
Test shape: (162, 1024)


In [16]:
# ------------------------------------------------------------
# Sample weights for TRAINING rows only
# ------------------------------------------------------------

sample_weight = np.ones(len(df_train))

sample_weight[df_train["data_source"] == "base_synthetic"] = 1.0
sample_weight[df_train["data_source"] == "contrastive_synthetic"] = 1.0
sample_weight[df_train["data_source"] == "contrastive_synthetic_v2"] = 0.8
sample_weight[df_train["data_source"] == "extra_institutionalized"] = 1.0

In [20]:
# ------------------------------------------------------------
# Fit SVD on train only, then transform train and test
# ------------------------------------------------------------

final_svd = TruncatedSVD(
    n_components=N_SVD_COMPONENTS,
    random_state=RANDOM_STATE
)

X_train = final_svd.fit_transform(X_train_embed)
X_test = final_svd.transform(X_test_embed)

In [21]:
# ------------------------------------------------------------
# Train Logistic Regression
# ------------------------------------------------------------

final_clf = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(
        C=0.5,
        max_iter=3000,
        solver="lbfgs",
        class_weight=None,
        random_state=RANDOM_STATE
    ))
])

final_clf.fit(
    X_train,
    y_train,
    logreg__sample_weight=sample_weight
)

print("Model trained on training split.")
print("Classes:", final_clf.named_steps["logreg"].classes_)

Model trained on training split.
Classes: ['ideological' 'institutionalized' 'internalized' 'interpersonal']


In [22]:
# ------------------------------------------------------------
# Evaluate on test set
# ------------------------------------------------------------

y_pred = final_clf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Macro F1:", f1_score(y_test, y_pred, average="macro"))

print("\nClassification report:")
print(classification_report(y_test, y_pred))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred, labels=final_clf.named_steps["logreg"].classes_))

Accuracy: 0.8518518518518519
Macro F1: 0.8448306806515762

Classification report:
                   precision    recall  f1-score   support

      ideological       0.75      0.73      0.74        41
institutionalized       0.98      0.92      0.95        51
     internalized       0.75      0.87      0.81        31
    interpersonal       0.89      0.87      0.88        39

         accuracy                           0.85       162
        macro avg       0.84      0.85      0.84       162
     weighted avg       0.86      0.85      0.85       162


Confusion matrix:
[[30  1  7  3]
 [ 4 47  0  0]
 [ 3  0 27  1]
 [ 3  0  2 34]]


In [25]:
# ------------------------------------------------------------
# Evaluate Top-2 accuracy
# ------------------------------------------------------------

proba = final_clf.predict_proba(X_test)

classes = final_clf.named_steps["logreg"].classes_

top2_acc = top_k_accuracy_score(
    y_test,
    proba,
    k=2,
    labels=classes
)

print("Top-2 Accuracy:", top2_acc)

Top-2 Accuracy: 0.9753086419753086
